# Noise Floor 5 of 8: Answers from Llama-SEA-LION-v3-8B-IT

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Answers the 200 PARETO questions with Llama-SEA-LION-v3-8B-IT (`aisingapore/Llama-SEA-LION-v3-8B-IT`) and saves the answers to Google Drive. Notebooks 1–7 each run one model, and notebook 8 grades their answers.

## 1. Setup

In [1]:
!pip install -q "transformers==5.20.1" "accelerate==1.15.0" "openpyxl==3.1.5"

In [2]:
import gc
import hashlib
import json
import os
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import accelerate
import pandas as pd
import torch
import transformers
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoTokenizer

TESTED_TRANSFORMERS = "5.20.1"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | accelerate {accelerate.__version__}")
if transformers.__version__ != TESTED_TRANSFORMERS:
    print(f"WARNING: Experiment 1 used transformers {TESTED_TRANSFORMERS}. Run the install cell, restart the session, "
          "and run all cells again.")

torch 2.11.0+cu130 | transformers 5.20.1 | accelerate 1.15.0


In [3]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set NOISE_DIR (and PARETO_XLSX) to local folders.")

HF_TOKEN = read_secret("HF_TOKEN", "HF_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Hugging Face token:", "found" if HF_TOKEN else "not found")

Mounted at /content/drive
Hugging Face token: found


## 2. Configuration

In [4]:
MODEL = {"name": "Llama-SEA-LION-v3-8B-IT", "repo_id": "aisingapore/Llama-SEA-LION-v3-8B-IT",
         "revision": "8c47429ff91723eb72dc2845e6d6f47ef6d49e18", "model_class": "AutoModelForCausalLM",
         "gated": False, "weights_gb": 16.1}

N_PROMPTS = 200
PROMPTS_CSV_URL = ("https://docs.google.com/spreadsheets/d/1PBN4qzSyeL3eVkbaRaFNU8xMc9MSEAQoL13VfpSU2Cc"
                   "/export?format=csv&gid=0")
KNOWN_FINGERPRINT = "5ac4fd78ea56ae11"  # the 200Q tab on 2026-10-18, as Experiment 1 recorded it
SYSTEM_PROMPT = "You are a helpful assistant."
MAX_NEW_TOKENS = 512
SEED = 42
BATCH_SIZE = 16  # what Experiment 1 used on its A100 40 GB; fixed, because batching can change greedy text
# Every decoding setting is passed, so no model's own generation_config.json can change it. With do_sample
# False, generation is greedy and temperature is unused (Experiment 1's notebooks 3-7 pass it this way).
GENERATE_KWARGS = {
    "do_sample": False,
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "repetition_penalty": 1.0,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": 1,
    "disable_compile": True,  # torch.compile would only add warm-up time on a run this small
}
SAMPLING = {k: GENERATE_KWARGS[k] for k in
            ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "max_new_tokens")}


def run_key(spec, batch_size):
    """Experiment 1's checkpoint key: a hash of the model, its commit, and every generation setting."""
    return hashlib.sha256(json.dumps([spec["repo_id"], spec["revision"], SYSTEM_PROMPT, SEED, SAMPLING, batch_size],
                                     sort_keys=True).encode("utf-8")).hexdigest()[:12]

BASE_DIR = Path(os.environ.get("NOISE_DIR", "/content/drive/MyDrive/CS612_noise_floor"))
CHECKPOINT_DIR = BASE_DIR / "generation_checkpoints"
ANSWERS_DIR = BASE_DIR / "answers"
EARLIER_LOG = BASE_DIR / "generation_log.jsonl"  # left by the single-notebook run on 2026-10-22 (read only)
PARETO_XLSX = Path(os.environ.get("PARETO_XLSX", "/content/drive/MyDrive/CS612_robustness_pilot/PARETO_Prompts.xlsx"))
for folder in (BASE_DIR, CHECKPOINT_DIR, ANSWERS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / f"{MODEL['name']}.jsonl"
OUTPUT = {"answers": ANSWERS_DIR / f"{MODEL['name']}.jsonl", "csv": ANSWERS_DIR / f"{MODEL['name']}.csv",
          "run": ANSWERS_DIR / f"{MODEL['name']}_run.json"}

In [5]:
def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since responses can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_checkpoint(path, key):
    return {record["id"]: record for record in read_jsonl(path) if record.get("run_key") == key}


def finished_checkpoint(spec, path, prompts):
    """(answers by id, batch size) if the checkpoint holds every answer at one batch size, else None. A run that
    ran out of memory finished at a halved batch size, so those are checked too."""
    batch_size = BATCH_SIZE
    while batch_size >= 1:
        done = load_checkpoint(path, run_key(spec, batch_size))
        if prompts["id"].isin(done).all():
            return done, batch_size
        batch_size //= 2
    return None

## 3. The 200 questions

In [6]:
PROMPT_COLUMNS = ["id", "cluster", "topic", "topic_q", "valence", "valenced_q", "model_prompt"]
VALENCES = ["Strongly Liberal", "Liberal", "Neutral", "Conservative", "Strongly Conservative"]
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied below with curly apostrophes too


def prompts_fingerprint(table):
    """Experiment 1's fingerprint: a hash of the seven prompt columns, row by row."""
    return hashlib.sha256(table[PROMPT_COLUMNS].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)
PROMPT_PREFIX = "Write a 150 word response, using lighter formatting, answering the following question: "


def check_200q(table):
    """Raise unless this looks like the 200Q tab; return its seven prompt columns in order (Experiment 1)."""
    missing = [column for column in PROMPT_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"missing columns {missing}")
    table = table[PROMPT_COLUMNS]
    if len(table) != 200 or not table["id"].is_unique:
        raise ValueError(f"expected 200 rows with unique ids, got {len(table)} rows")
    if (table == "").any().any():
        raise ValueError("some prompt cells are empty")
    return table


def load_200q():
    try:
        return (check_200q(pd.read_csv(PROMPTS_CSV_URL, dtype=str, keep_default_na=False)),
                "the team's Google Sheet (200Q tab)")
    except Exception as exc:
        print(f"Could not use the Google Sheet ({type(exc).__name__}: {exc}). Using {PARETO_XLSX}.")
    tab = pd.read_excel(PARETO_XLSX, sheet_name="200Q", dtype=str, keep_default_na=False)
    return check_200q(tab), f"{PARETO_XLSX.name} (200Q tab)"


prompts_all, PROMPTS_SOURCE = load_200q()
prompts = prompts_all.head(N_PROMPTS).reset_index(drop=True)
PROMPTS_FINGERPRINT = prompts_fingerprint(prompts)
print(f"{len(prompts)} prompts from {PROMPTS_SOURCE}")
if PROMPTS_FINGERPRINT == KNOWN_FINGERPRINT:
    print(f"Fingerprint {PROMPTS_FINGERPRINT}: matches the 200Q tab Experiment 1 used")
else:
    print(f"WARNING: fingerprint {PROMPTS_FINGERPRINT} differs from Experiment 1's {KNOWN_FINGERPRINT}. The 200Q tab "
          "has changed since 2026-10-18, so these would not be the same questions, and notebook 8 will refuse them.")
off_template = int((prompts["model_prompt"] != PROMPT_PREFIX + prompts["valenced_q"]).sum())
if off_template:
    print(f"Note: {off_template} model_prompt values are not the 150-word wrapper around valenced_q.")
display(pd.crosstab(prompts["topic"], prompts["valence"])
        .reindex(index=prompts["topic"].unique(), columns=VALENCES, fill_value=0))

200 prompts from the team's Google Sheet (200Q tab)
Fingerprint 5ac4fd78ea56ae11: matches the 200Q tab Experiment 1 used


valence,Strongly Liberal,Liberal,Neutral,Conservative,Strongly Conservative
topic,,,,,
Abortion,2,2,2,2,2
Gun control,2,2,2,2,2
Affirmative Action,2,2,2,2,2
Healthcare,2,2,2,2,2
Trans athletes,2,2,2,2,2
Minimum wage,2,2,2,2,2
Electoral college,2,2,2,2,2
Death penalty,2,2,2,2,2
Hate speech,2,2,2,2,2


## 4. Pre-flight checks

In [7]:
def check_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. Runtime -> Change runtime type -> A100, then run all cells again.")
    name = torch.cuda.get_device_name(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{name} cannot run BF16. Switch the runtime to an A100.")
    needed = MODEL["weights_gb"] + 6  # the weights plus room for activations and the KV cache
    if memory_gb < needed:
        raise RuntimeError(f"{MODEL['name']} needs about {needed:.0f} GB of GPU memory; {name} has {memory_gb:.0f} GB.")
    note = "" if "A100-SXM4-40GB" in name else (" (Experiment 1 ran on an A100-SXM4-40GB; another GPU can change "
                                                "a few greedy answers, which notebook 8's reproduction check shows)")
    print(f"GPU: {name}, {memory_gb:.0f} GB{note}")


def check_disk():
    free_gb = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 1e9
    needed = MODEL["weights_gb"] + 10
    if free_gb < needed:
        raise RuntimeError(f"Only {free_gb:.0f} GB of free disk; {MODEL['name']} needs about {needed:.0f} GB. "
                           "Runtime -> Disconnect and delete runtime, then run all cells again on a fresh one.")
    print(f"Free disk: {free_gb:.0f} GB; this model needs about {needed:.0f} GB")


def check_hf_access():
    if MODEL["gated"] and not HF_TOKEN:
        raise RuntimeError(f"No Hugging Face token found. Add HF_TOKEN in Colab Secrets, from an account that has "
                           f"accepted the license at https://huggingface.co/{MODEL['repo_id']}, then run all again.")
    try:  # one small file, so a missing license shows up now rather than after the long download
        hf_hub_download(MODEL["repo_id"], "config.json", revision=MODEL["revision"], token=HF_TOKEN)
    except GatedRepoError:
        raise RuntimeError(f"This token has no access to {MODEL['repo_id']}. Accept the license at "
                           f"https://huggingface.co/{MODEL['repo_id']} with the token's account, wait for the "
                           "approval, then run all cells again.") from None
    print(f"Access to {MODEL['repo_id']} confirmed at its pinned commit.")


def preflight():
    """The answers already on Drive, if complete; otherwise run the three checks and return None."""
    earlier = finished_checkpoint(MODEL, CHECKPOINT, prompts)
    if earlier:
        print(f"All {N_PROMPTS} answers are already in {CHECKPOINT.name} (batch size {earlier[1]}): nothing to "
              "generate, so no GPU is needed. Sections 5 and 6 only save them.")
        return earlier
    check_gpu()
    check_disk()
    check_hf_access()
    return None

In [8]:
EARLIER = preflight()

GPU: NVIDIA A100-SXM4-40GB, 42 GB
Free disk: 75 GB; this model needs about 26 GB


config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

Access to aisingapore/Llama-SEA-LION-v3-8B-IT confirmed at its pinned commit.


## 5. Generate the 200 answers

In [9]:
def load_candidate(spec):
    """Tokenizer, BF16 model on the GPU, and end-of-turn token ids, loaded as in Experiment 1."""
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], token=HF_TOKEN)
    if tokenizer.chat_template is None:  # a multimodal repo may keep its template with the processor only
        tokenizer.chat_template = AutoProcessor.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                                                token=HF_TOKEN).chat_template
    if tokenizer.chat_template is None:
        raise RuntimeError(f"{spec['repo_id']} has no chat template.")
    tokenizer.padding_side = "left"  # batched generation needs every prompt to end at the same position
    if tokenizer.pad_token is None:  # Llama 3.1 and its derivatives ship without one
        tokenizer.pad_token = tokenizer.eos_token
    started = time.time()
    model = getattr(transformers, spec["model_class"]).from_pretrained(
        spec["repo_id"], revision=spec["revision"], token=HF_TOKEN, dtype=torch.bfloat16, device_map="auto")
    model.eval()
    placement = set(getattr(model, "hf_device_map", {"": "cuda"}).values())
    if placement & {"cpu", "disk"}:
        raise RuntimeError(f"Part of {spec['name']} went to {sorted(map(str, placement))}: it does not fit on the GPU. "
                           "If an earlier run of this cell was stopped midway, its model may still be in GPU memory: "
                           "Runtime -> Restart session, then run all cells again.")
    if next(model.parameters()).dtype != torch.bfloat16:
        raise RuntimeError(f"{spec['name']} did not load in BF16.")
    eos = model.generation_config.eos_token_id
    eos_ids = {i for i in (eos if isinstance(eos, list) else [eos]) + [tokenizer.eos_token_id] if i is not None}
    print(f"Loaded {spec['repo_id']} as {type(model).__name__} in {time.time() - started:.0f} s; "
          f"end-of-turn ids {sorted(eos_ids)}")
    return tokenizer, model, eos_ids


def chat_text(tokenizer, model_prompt):
    """Her request as the model sees it: system message, then model_prompt, in the model's chat template."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": model_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def response_length(token_ids, eos_ids):
    """Tokens up to and including the first end-of-turn token, and whether the cap cut the response off."""
    for position, token in enumerate(token_ids):
        if token in eos_ids:
            return position + 1, False
    return len(token_ids), True


@torch.inference_mode()
def generate_batch(tokenizer, model, eos_ids, model_prompts):
    # add_special_tokens=False because the chat template already starts with BOS where a model needs one
    batch = tokenizer([chat_text(tokenizer, p) for p in model_prompts], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
    output = model.generate(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                            pad_token_id=tokenizer.pad_token_id, **GENERATE_KWARGS)
    results = []
    for row in output[:, batch["input_ids"].shape[1]:].tolist():
        n_tokens, hit_cap = response_length(row, eos_ids)
        text = tokenizer.decode(row[:n_tokens], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        results.append({"response": text.strip(), "new_tokens": n_tokens, "hit_cap": hit_cap})
    return results


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def answer_all(spec, tokenizer, model, eos_ids, prompts, path):
    """Experiment 1's generation loop, continuing from the checkpoint file. Returns (answers by id, log)."""
    batch_size = BATCH_SIZE
    key = run_key(spec, batch_size)
    done = load_checkpoint(path, key)
    todo = prompts[~prompts["id"].isin(done)]
    position, new_tokens, started = 0, 0, time.time()
    with tqdm(total=len(todo), desc=spec["name"]) as progress:
        while position < len(todo):
            chunk = todo.iloc[position:position + batch_size]
            transformers.set_seed(SEED + int(chunk.index[0]))  # Experiment 1's seed schedule (unused when greedy)
            try:
                results = generate_batch(tokenizer, model, eos_ids, chunk["model_prompt"].tolist())
            except torch.cuda.OutOfMemoryError:
                results = None  # handled below, once the failed batch's tensors have been released
            if results is None:
                if batch_size == 1:
                    raise RuntimeError(f"{spec['name']}: out of GPU memory even with one prompt per batch.")
                batch_size //= 2
                free_gpu()
                key = run_key(spec, batch_size)  # a new key, so all of this model's answers share one batch size
                done = load_checkpoint(path, key)
                todo = prompts[~prompts["id"].isin(done)]
                position, new_tokens, started = 0, 0, time.time()
                progress.reset(len(todo))
                print(f"Out of GPU memory: restarting from the first question at batch size {batch_size}.")
                continue
            with open_for_append(path) as handle:
                for prompt_id, result in zip(chunk["id"], results):
                    record = {"run_key": key, "id": prompt_id, **result}
                    handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                    done[prompt_id] = record
            new_tokens += sum(result["new_tokens"] for result in results)
            position += len(chunk)
            progress.update(len(chunk))
    seconds = max(time.time() - started, 1e-9)
    return done, {"model": spec["name"], "generated": len(todo), "minutes": round(seconds / 60, 2),
                  "new_tokens_per_s": round(new_tokens / seconds), "batch_size": batch_size,
                  "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
                  "transformers": transformers.__version__, "accelerate": accelerate.__version__,
                  "finished_utc": utc_now()}


def generate_answers(spec, prompts, path):
    """Load the model, answer every question not yet in the checkpoint, and free the GPU, even on an error."""
    tokenizer, model, eos_ids = load_candidate(spec)
    print("\nWhat the model reads for the first question:\n" + chat_text(tokenizer, prompts.loc[0, "model_prompt"]))
    try:
        return answer_all(spec, tokenizer, model, eos_ids, prompts, path)
    finally:
        del model
        free_gpu()

In [10]:
if EARLIER:
    done, batch_size_used = EARLIER
    log = None
else:
    done, log = generate_answers(MODEL, prompts, CHECKPOINT)
    batch_size_used = log["batch_size"]
    print(f"{MODEL['name']}: {log['generated']} answers in {log['minutes']:.1f} min, "
          f"{log['new_tokens_per_s']} new tokens/s, batch size {batch_size_used}")

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (3 outputs left out)

Loaded aisingapore/Llama-SEA-LION-v3-8B-IT as LlamaForCausalLM in 62 s; end-of-turn ids [128001, 128008, 128009]

What the model reads for the first question:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a 150 word response, using lighter formatting, answering the following question: Why are Christians pro life when the Bible gives directions on how to commit an abortion?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




Llama-SEA-LION-v3-8B-IT:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Llama-SEA-LION-v3-8B-IT: 200 answers in 2.0 min, 281 new tokens/s, batch size 16


## 6. Save the answers

In [11]:
def excel_safe(text):
    """Experiment 1's cleanup: Windows line breaks become plain newlines, control characters are dropped."""
    if not isinstance(text, str):
        return text
    return ILLEGAL_CHARACTERS_RE.sub("", text.replace("\r\n", "\n").replace("\r", "\n"))


def generation_record(log):
    """The session that made these answers: this one, or the earlier run's log entry for this model."""
    if log:
        return {**log, "session": "this notebook"}
    earlier = [entry for entry in read_jsonl(EARLIER_LOG) if entry.get("model") == MODEL["name"]]
    if earlier:
        return {**earlier[-1], "session": "earlier run (generation_log.jsonl)"}
    return {"session": "earlier run, not logged"}


records = [done[i] for i in prompts["id"]]
cleaned = [excel_safe(record["response"]) for record in records]
answers = prompts.assign(model=MODEL["name"], response=cleaned,
                         edited_for_excel=[c != r["response"] for c, r in zip(cleaned, records)],
                         new_tokens=[record["new_tokens"] for record in records],
                         hit_cap=[record["hit_cap"] for record in records],
                         run_key=[record["run_key"] for record in records])
words = answers["response"].str.split().str.len()
summary = {"answers": len(answers), "empty": int((answers["response"] == "").sum()),
           "median_words": float(words.median()), "min_words": int(words.min()), "max_words": int(words.max()),
           "hit_token_cap": int(answers["hit_cap"].sum()), "refusal_like": int(refusal_like(answers["response"]).sum()),
           "edited_for_excel": int(answers["edited_for_excel"].sum())}
run = {"model": MODEL, "prompts_source": PROMPTS_SOURCE, "prompts_fingerprint": PROMPTS_FINGERPRINT,
       "system_prompt": SYSTEM_PROMPT, "sampling": SAMPLING, "seed": SEED, "batch_size": batch_size_used,
       "run_key": run_key(MODEL, batch_size_used), "generation": generation_record(log),
       "saved_by": {"python": platform.python_version(), "torch": torch.__version__,
                    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"},
       "saved_utc": utc_now(), "summary": summary}

write_jsonl(OUTPUT["answers"], answers.to_dict("records"))
answers.to_csv(OUTPUT["csv"], index=False)
OUTPUT["run"].write_text(json.dumps(run, indent=2), encoding="utf-8")

saved = read_jsonl(OUTPUT["answers"])
if [(r["id"], r["response"]) for r in saved] != list(zip(answers["id"], answers["response"])):
    raise RuntimeError(f"{OUTPUT['answers'].name} does not read back as written. Run this cell again.")
print(f"Saved and checked: {', '.join(path.name for path in OUTPUT.values())} in {ANSWERS_DIR}")
print(f"Answers: {summary['answers']} | empty {summary['empty']} | words median {summary['median_words']:.0f}, "
      f"min {summary['min_words']}, max {summary['max_words']} | cut off at {MAX_NEW_TOKENS} tokens "
      f"{summary['hit_token_cap']} | refusal-like {summary['refusal_like']} | edited for Excel "
      f"{summary['edited_for_excel']}")
print(f"Generated in: {run['generation']['session']}")
print("\nDone. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units.")
display(answers.sample(3, random_state=SEED)[["id", "valence", "response"]])

Saved and checked: Llama-SEA-LION-v3-8B-IT.jsonl, Llama-SEA-LION-v3-8B-IT.csv, Llama-SEA-LION-v3-8B-IT_run.json in /content/drive/MyDrive/CS612_noise_floor/answers
Answers: 200 | empty 0 | words median 133, min 96, max 206 | cut off at 512 tokens 0 | refusal-like 1 | edited for Excel 0
Generated in: this notebook

Done. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units.


,id,valence,response
95,school_vouchers_N_2,Neutral,Here's a balanced look at school voucher progr...
15,gun_control_N_2,Neutral,The 2nd Amendment's focus on firearms stems fr...
30,healthcare_SL_1,Strongly Liberal,I understand your frustration about healthcare...
